# 인공신경망 기초

In [ ]:
#1단계
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print("훈련:", X_train_s.shape, "/ 테스트:", X_test_s.shape)
print("특성 개수:", X.shape [1])

훈련: (455, 30) / 테스트: (114, 30)
특성 개수: 30


In [ ]:
#2단계
from sklearn.linear_model import LogisticRegression

logit = LogisticRegression(max_iter=1000).fit(X_train_s, y_train)
w, b = logit.coef_[0], logit.intercept_[0] # 가중치 벡터와 편향

def sigmoid(z):
  return 1 / (1 + np.exp(-z))

x3 = X_test_s[3] # 확률이 0.5에 가장 가까운 환자
z3 = np.dot(w, x3) + b #가중합
a3 = sigmoid(z3) #활성화 함수를 통과

print("가중치 모양:", w.shape, "/ 편향:", round(b, 4))
print("가중합 z =", round(z3, 4), "/ 출력:", round(a3, 6))
print("사이킷런 확률:", round(logit.predict_proba(x3.reshape(1, -1))[0, 1], 6))

가중치 모양: (30,) / 편향: 0.3022
가중합 z = 0.1342 / 출력: 0.533509
사이킷런 확률: 0.533509


In [ ]:
#3단계. 활성홯 함수
def step(z):
  return (z>0).astype(float)

def relu(z):
  return np.maximum(0, z)

zs = np.array([-3.0, -1.0, 0.0, 1.0, 3.0])
print("z      :", zs)
print("step   :", step(zs))
print("sigmoid:", np.round(sigmoid(zs),4))
print("tanh   :", np.round(np.tanh(zs), 4))
print("relu   :", relu(zs))

z      : [-3. -1.  0.  1.  3.]
step   : [0. 0. 0. 1. 1.]
sigmoid: [0.0474 0.2689 0.5    0.7311 0.9526]
tanh   : [-0.9951 -0.7616  0.      0.7616  0.9951]
relu   : [0. 0. 0. 1. 3.]


In [ ]:
#4단계
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_and = np.array([0, 0, 0, 1])
y_or = np.array([0, 1, 1, 1])
y_xor = np.array([0, 1, 1, 0])

print("입력\n", X_xor)
print("AND:", y_and)
print("OR :", y_or)
print("XOR:", y_xor)

입력
 [[0. 0.]
 [0. 1.]
 [1. 0.]
 [1. 1.]]
AND: [0 0 0 1]
OR : [0 1 1 1]
XOR: [0 1 1 0]


In [ ]:
#5단계
from sklearn.linear_model import Perceptron
from sklearn.metrics import accuracy_score

for name, target in [("AND", y_and), ("OR", y_or), ("XOR", y_xor)]:
  p = Perceptron(max_iter=1000, tol=None, random_state=0)
  p.fit(X_xor, target)
  pred = p.predict(X_xor)
  print("%-4s 예측 %s 정답 %s 정확도 %.2f" % (
    name, pred, target, accuracy_score(target, pred)))

AND  예측 [0 0 0 1] 정답 [0 0 0 1] 정확도 1.00
OR   예측 [0 1 1 1] 정답 [0 1 1 1] 정확도 1.00
XOR  예측 [0 0 0 0] 정답 [0 1 1 0] 정확도 0.50


In [ ]:
#6단계
W1 = np.array([[1.0, 1.0], [1.0, 1.0]])
b1 = np.array([-0.5, -1.5])
W2 = np.array([[1.0], [-1.0]])
b2 = np.array([-0.5])
H = step(X_xor @ W1 + b1)
O = step(H @ W2 + b2)
print("입력", X_xor.shape, "가중치", W1.shape, "은닉 출력", H.shape)
print("은닉층 h1(OR), h2(AND):\n", H)
print("출력:", O.ravel(), "/ 정답:", y_xor)
print("정확도:", accuracy_score(y_xor, O.ravel()))


입력 (4, 2) 가중치 (2, 2) 은닉 출력 (4, 2)
은닉층 h1(OR), h2(AND):
 [[0. 0.]
 [1. 0.]
 [1. 0.]
 [1. 1.]]
출력: [0. 1. 1. 0.] / 정답: [0 1 1 0]
정확도: 1.0


In [ ]:
#7단계
from sklearn.neural_network import MLPClassifier

for h in [2, 4, 8]:
  ok = 0
  for seed in range(10):
    m = MLPClassifier(hidden_layer_sizes=(h,), activation="relu",
                      max_iter=5000, random_state=seed)
    m.fit(X_xor, y_xor)
    ok += int(accuracy_score(y_xor, m.predict(X_xor)) == 1.0)
  print("은닉 노드 %d개 : 난수 씨앗 열 번 중 %d번 성공" % (h, ok))

은닉 노드 2개 : 난수 씨앗 열 번 중 1번 성공
은닉 노드 4개 : 난수 씨앗 열 번 중 6번 성공
은닉 노드 8개 : 난수 씨앗 열 번 중 10번 성공


In [ ]:
#8단계
rng = np.random.RandomState(42)
W1 = rng.randn(30, 16) * 0.1
b1 = np.zeros(16)
W2 = rng.randn(16, 1) * 0.1
b2 = np.zeros(1)

print("X_train_s      :", X_train_s.shape)
print("W1             :", W1.shape, " b1:", b1.shape)
print("X @ W1         :", (X_train_s @ W1).shape)
print("W2             :", W2.shape, " b2:", b2.shape)
print("(X @ W1) @ W2  :", (X_train_s @ W1 @ W2).shape)

X_train_s      : (455, 30)
W1             : (30, 16)  b1: (16,)
X @ W1         : (455, 16)
W2             : (16, 1)  b2: (1,)
(X @ W1) @ W2  : (455, 1)


In [ ]:
#9단계
def forward(X, W1, b1, W2, b2, act=relu):
  H = act(X @ W1 + b1)
  Z = H @ W2 + b2
  return sigmoid(Z), H

proba, H = forward(X_train_s, W1, b1, W2, b2)
print("은닉층 출력:", H.shape, "/ 최종 출력:", proba.shape)
print("은닉층에서 0이 된 칸의 비율:", round(float((H == 0).mean()), 4))
print("출력 확률 앞 다섯 개:", np.round(proba[:5].ravel(), 4))
print("가중치 %d개 + 편향 %d개 = 파라미터 %d개" % (
    W1.size + W2.size, b1.size + b2.size,
    W1.size + b1.size + W2.size + b2.size))

은닉층 출력: (455, 16) / 최종 출력: (455, 1)
은닉층에서 0이 된 칸의 비율: 0.5058
출력 확률 앞 다섯 개: [0.509  0.469  0.5141 0.482  0.5133]
가중치 496개 + 편향 17개 = 파라미터 513개
